In [1]:
!pip install wfdb torchmetrics

In [2]:
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
from sklearn.preprocessing import MultiLabelBinarizer
import torchmetrics
import os

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [15]:
if device.type == 'cuda':
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
if device.type == 'cuda':
    !time unzip -q "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip" -d "/content/"
    !ls -lah /content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 | head


real	0m12.220s
user	0m8.365s
sys	0m3.070s
total 16M
drwxrwxrwx  3 root root 4.0K Mar 12 04:23 .
drwxr-xr-x  1 root root 4.0K Mar 12 04:46 ..
-rw-rw-rw-  1 root root 1.4K Mar  9 15:42 example_physionet.py
-rw-rw-rw-  1 root root  15K Mar  9 15:42 LICENSE.txt
-rw-rw-rw-  1 root root 6.3M Mar  9 15:42 ptbxl_database.csv
-rw-rw-rw-  1 root root 6.0K Mar  9 15:42 ptbxl_v102_changelog.txt
-rw-rw-rw-  1 root root 7.3K Mar  9 15:42 ptbxl_v103_changelog.txt
-rw-rw-rw-  1 root root 1.1M Mar  9 15:42 RECORDS
drwxrwxrwx 24 root root 4.0K Mar  9 15:48 records100


In [17]:
def load_raw_data(df, sampling_rate):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

if device.type == 'cuda':
    parent_path = '/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
else:
    parent_path = os.curdir
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(os.path.join(parent_path,'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(os.path.join(parent_path,'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv


In [18]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

In [19]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

In [20]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

((19601, 1000, 12), dtype('float64'), (19601, 5), dtype('int64'))

In [ ]:
# just examining the data some
summed = np.sum(y_train, axis=1)
record_ids_with_no_labels_true = []
for id, class_sum in enumerate(summed):
    if class_sum == 0:
        record_ids_with_no_labels_true.append(id)
print(f'{len(record_ids_with_no_labels_true)} train examples have all labels as "0"')

371 train examples have all labels as "0"


In [22]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).transpose(1,2)  # the 12 channels dimension is initially loaded in as last
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32).transpose(1,2)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [23]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [24]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(12, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)

        # average the whole time series into 1 length to account for any time series length
        # while this discards temporal detail, the abnormalities are more concerned over the presence/strength of extracted features (representing the presence of abnormalities)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [25]:
loss_fn = nn.BCEWithLogitsLoss()  # TODO: try adding pos_weight with pos_weight_c = N_neg_c / N_pos_c (where c is each of the 5 labels)

In [27]:
model = CNN(num_classes=y_train_tensor.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_loss = float('inf')  # start with infinity
if device.type == 'cuda':
    best_model_save_dir = '/content/drive/MyDrive/'
else:
    best_model_save_dir = 'models'
best_model_save_path = os.path.join(best_model_save_dir, "cnn_best_model.pt")

for epoch in range(20):
    running_loss = 0.0
    model.train()

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        preds = model(x)

        loss = loss_fn(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), best_model_save_path)

Epoch 1, Loss: 0.3632
Epoch 2, Loss: 0.3119
Epoch 3, Loss: 0.2949
Epoch 4, Loss: 0.2862
Epoch 5, Loss: 0.2808
Epoch 6, Loss: 0.2738
Epoch 7, Loss: 0.2708
Epoch 8, Loss: 0.2660
Epoch 9, Loss: 0.2623
Epoch 10, Loss: 0.2603
Epoch 11, Loss: 0.2573
Epoch 12, Loss: 0.2534
Epoch 13, Loss: 0.2514
Epoch 14, Loss: 0.2496
Epoch 15, Loss: 0.2466
Epoch 16, Loss: 0.2441
Epoch 17, Loss: 0.2421
Epoch 18, Loss: 0.2403
Epoch 19, Loss: 0.2381
Epoch 20, Loss: 0.2362


In [28]:
model.eval()
all_test_preds, all_test_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_test_preds.append(preds.cpu())
        all_test_labels.append(y.cpu())

# Concatenate all batches
all_test_preds = torch.cat(all_test_preds, dim=0)
all_test_labels = torch.cat(all_test_labels, dim=0)

In [32]:
# Per-class accuracy
per_class_acc = (all_test_preds == all_test_labels).float().mean(dim=0)  # [num_classes]
# Print results
for idx, acc in enumerate(per_class_acc):
    print(f"Class {mlb.classes_[idx]} accuracy: {acc.item():.3f}")

Class CD accuracy: 0.887
Class HYP accuracy: 0.913
Class MI accuracy: 0.867
Class NORM accuracy: 0.865
Class STTC accuracy: 0.891


In [33]:
from sklearn.metrics import f1_score
f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
for idx, score in enumerate(f1_scores):
    print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")

Class CD F1-score: 0.713
Class HYP F1-score: 0.600
Class MI F1-score: 0.734
Class NORM F1-score: 0.851
Class STTC F1-score: 0.764


In [34]:
from sklearn.metrics import roc_auc_score

num_classes = all_test_labels.shape[1]
aucs = []

for i in range(num_classes):
    try:
        auc = roc_auc_score(all_test_labels[:, i], all_test_preds[:, i])
    except ValueError:
        # happens if class i has no positive samples in test set
        auc = float('nan')
    aucs.append(auc)

for idx, auc in enumerate(aucs):
    print(f"Class {mlb.classes_[idx]} AUROC: {auc:.3f}")

Class CD AUROC: 0.794
Class HYP AUROC: 0.756
Class MI AUROC: 0.822
Class NORM AUROC: 0.866
Class STTC AUROC: 0.841
